# ◆ Studio — Free Image + Video Generator (SDXL · FLUX · LTX · Wan)

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Press ▶️ below and wait ~10 minutes.
3. Type a prompt, pick a style, press **Generate**.

Free Colab gives a few GPU hours per day. When it stops, press play again later.

In [ ]:
#@title ▶️ Press play. Wait ~10 minutes. The studio appears below.
import subprocess, os, gc, time, random
print("1/3 Installing ...")
subprocess.run("pip install -q -U diffusers ftfy transformers accelerate bitsandbytes sentencepiece gradio imageio imageio-ffmpeg protobuf", shell=True, check=True)

import torch, gradio as gr
from diffusers import StableDiffusionXLPipeline, LTXPipeline
from diffusers.utils import export_to_video
from transformers import T5EncoderModel, BitsAndBytesConfig

os.makedirs("/content/outputs", exist_ok=True)
state = {"name": None, "pipe": None}
history = []

def free():
    state["pipe"] = None; state["name"] = None
    gc.collect(); torch.cuda.empty_cache()

def get_image_pipe(model):
    if state["name"] != model:
        free()
        if model == "SDXL":
            p = StableDiffusionXLPipeline.from_pretrained(
                "stabilityai/stable-diffusion-xl-base-1.0",
                torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
            p.enable_vae_slicing()
        else:  # FLUX.1 schnell, 4-bit so it fits the free GPU
            from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DBnB
            rid = "black-forest-labs/FLUX.1-schnell"
            te2 = T5EncoderModel.from_pretrained(rid, subfolder="text_encoder_2",
                    quantization_config=BitsAndBytesConfig(load_in_8bit=True), torch_dtype=torch.float16)
            tr = FluxTransformer2DModel.from_pretrained(rid, subfolder="transformer",
                    quantization_config=DBnB(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
                    torch_dtype=torch.float16)
            p = FluxPipeline.from_pretrained(rid, text_encoder_2=te2, transformer=tr, torch_dtype=torch.float16)
            p.enable_model_cpu_offload()
            p.vae.enable_slicing(); p.vae.enable_tiling()
        state["pipe"], state["name"] = p, model
    return state["pipe"]

def get_video_pipe(model):
    if state["name"] != model:
        free()
        if model == "LTX-Video":
            te = T5EncoderModel.from_pretrained("Lightricks/LTX-Video", subfolder="text_encoder",
                    quantization_config=BitsAndBytesConfig(load_in_8bit=True), torch_dtype=torch.float16)
            p = LTXPipeline.from_pretrained("Lightricks/LTX-Video", text_encoder=te, torch_dtype=torch.float16)
        else:  # Wan 2.1 1.3B
            from diffusers import WanPipeline, AutoencoderKLWan
            from transformers import UMT5EncoderModel
            rid = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
            te = UMT5EncoderModel.from_pretrained(rid, subfolder="text_encoder",
                    quantization_config=BitsAndBytesConfig(load_in_8bit=True), torch_dtype=torch.float16)
            vae = AutoencoderKLWan.from_pretrained(rid, subfolder="vae", torch_dtype=torch.float32)
            p = WanPipeline.from_pretrained(rid, text_encoder=te, vae=vae, torch_dtype=torch.bfloat16)
        p.enable_model_cpu_offload()
        try: p.vae.enable_tiling()
        except Exception: pass
        state["pipe"], state["name"] = p, model
    return state["pipe"]

STYLES = {
    "None": ("", ""),
    "Photorealistic": ("professional photograph, 85mm lens, natural lighting, highly detailed, sharp focus", "painting, drawing, cartoon, cgi"),
    "Cinematic": ("cinematic film still, dramatic lighting, shallow depth of field, color graded, 35mm", "cartoon, flat, low contrast"),
    "Anime": ("anime illustration, clean line art, vibrant colors, studio quality", "photo, realistic, 3d"),
    "3D Render": ("3d render, octane, unreal engine 5, global illumination, ultra detailed", "photo, sketch, flat"),
    "Watercolor": ("watercolor painting, soft edges, paper texture, delicate brush strokes", "photo, 3d, sharp"),
    "Digital Art": ("digital painting, concept art, trending on artstation, dramatic composition", "photo, blurry"),
}
SIZES = {"Square 1:1": (1024, 1024), "Portrait 3:4": (896, 1152), "Landscape 4:3": (1152, 896), "Widescreen 16:9": (1344, 768)}
BASE_NEG = "blurry, low quality, distorted, deformed, watermark, text, logo"

def build(prompt, style):
    s, n = STYLES[style]
    return (f"{prompt}, {s}" if s else prompt), (f"{BASE_NEG}, {n}" if n else BASE_NEG)

def make_image(model, prompt, style, size, count, steps, seed, progress=gr.Progress()):
    if not prompt.strip():
        raise gr.Error("Type a description first.")
    progress(0, desc=f"Loading {model} (first time downloads the model)" if state["name"] != model else "Starting")
    p = get_image_pipe(model)
    pos, neg = build(prompt, style)
    w, h = SIZES[size]
    seed = int(seed) if int(seed) >= 0 else random.randint(0, 2**31)
    gen = torch.Generator("cuda").manual_seed(seed)
    t0 = time.time()
    if model == "FLUX.1 schnell":
        steps = 4
        def cb(pipe, i, t, kw):
            progress((i + 1) / steps, desc=f"Rendering step {i+1}/{steps}"); return kw
        imgs = p(prompt=pos, num_inference_steps=steps, guidance_scale=0.0, max_sequence_length=256,
                 width=w, height=h, num_images_per_prompt=int(count), generator=gen, callback_on_step_end=cb).images
    else:
        def cb(pipe, i, t, kw):
            progress((i + 1) / steps, desc=f"Rendering step {i+1}/{steps}"); return kw
        imgs = p(prompt=pos, negative_prompt=neg, num_inference_steps=int(steps), guidance_scale=7,
                 width=w, height=h, num_images_per_prompt=int(count), generator=gen, callback_on_step_end=cb).images
    for i, im in enumerate(imgs):
        path = f"/content/outputs/img_{int(time.time())}_{i}.png"; im.save(path); history.insert(0, path)
    return imgs, history[:24], f"Done · {model} · seed {seed} · {w}×{h} · {time.time()-t0:.0f} s"

def make_video(model, prompt, style, seconds, steps, seed, progress=gr.Progress()):
    if not prompt.strip():
        raise gr.Error("Type a description first.")
    progress(0, desc=f"Loading {model} (first time downloads the model)" if state["name"] != model else "Starting")
    p = get_video_pipe(model)
    pos, neg = build(prompt, style)
    seed = int(seed) if int(seed) >= 0 else random.randint(0, 2**31)
    gen = torch.Generator("cuda").manual_seed(seed)
    t0 = time.time()
    def cb(pipe, i, t, kw):
        progress((i + 1) / steps, desc=f"Rendering step {i+1}/{steps}"); return kw
    if model == "LTX-Video":
        fps, w, h = 24, 768, 512
        frames = int(seconds) * fps + 1
        vid = p(prompt=pos, negative_prompt=neg, width=w, height=h, num_frames=frames,
                num_inference_steps=int(steps), guidance_scale=3, generator=gen, callback_on_step_end=cb).frames[0]
    else:
        fps, w, h = 16, 832, 480
        frames = int(seconds) * fps + 1
        vid = p(prompt=pos, negative_prompt=neg, width=w, height=h, num_frames=frames,
                num_inference_steps=int(steps), guidance_scale=5, generator=gen, callback_on_step_end=cb).frames[0]
    path = f"/content/outputs/video_{int(time.time())}.mp4"
    export_to_video(vid, path, fps=fps)
    return path, f"Done · {model} · seed {seed} · {seconds}s · {w}×{h} · {(time.time()-t0)/60:.1f} min"

CSS = """
.gradio-container {max-width: 1360px !important; margin: auto; padding-top: 0 !important;}
#topbar {display:flex; align-items:center; justify-content:space-between; padding:14px 6px 10px; border-bottom:1px solid #1f2937; margin-bottom:14px;}
#topbar .brand {display:flex; align-items:center; gap:10px; font-weight:700; font-size:1.15rem; letter-spacing:-.01em;}
#topbar .brand .mark {width:28px;height:28px;border-radius:8px;background:linear-gradient(135deg,#6366f1,#8b5cf6);display:inline-block;}
#topbar .brand small {font-weight:500; opacity:.55; font-size:.8rem; margin-left:6px;}
#topbar .badges {display:flex; gap:8px;}
.badge {font-size:.75rem; padding:4px 10px; border-radius:999px; border:1px solid #243044; background:#0f1522; color:#cbd5e1;}
.badge.ok {border-color:#14532d; color:#86efac; background:#052e16;}
.badge.ok:before {content:"●"; margin-right:6px; font-size:.6rem; vertical-align:middle;}
#panel {background:#121826; border:1px solid #1f2937; border-radius:14px; padding:14px;}
#panel .label-wrap span, #panel label span {font-size:.78rem; text-transform:uppercase; letter-spacing:.06em; color:#94a3b8;}
#prompt textarea {font-size:1rem; line-height:1.45;}
#go {height:50px; font-size:1rem; font-weight:600; letter-spacing:.01em;}
#canvas {background:#0f1522; border:1px solid #1f2937; border-radius:14px; padding:10px; min-height:620px;}
#status {font-size:.85rem; color:#94a3b8; padding:4px 6px;}
#recent .grid {gap:6px;}
.section-title {font-size:.78rem; text-transform:uppercase; letter-spacing:.06em; color:#94a3b8; margin:4px 0 6px;}
footer {display:none !important;}
"""
theme = gr.themes.Base(primary_hue="indigo", neutral_hue="slate", radius_size="md",
    font=gr.themes.GoogleFont("Inter"), font_mono=gr.themes.GoogleFont("JetBrains Mono")).set(
    body_background_fill="#0b0f19", body_background_fill_dark="#0b0f19",
    block_background_fill="#121826", block_background_fill_dark="#121826",
    block_border_width="0px", panel_background_fill="#121826", panel_background_fill_dark="#121826",
    input_background_fill="#0b111d", input_background_fill_dark="#0b111d",
    input_border_color="#243044", input_border_color_dark="#243044",
    body_text_color="#e5e7eb", body_text_color_dark="#e5e7eb",
    block_label_text_color="#94a3b8", block_label_text_color_dark="#94a3b8",
    block_title_text_color="#94a3b8", block_title_text_color_dark="#94a3b8",
    slider_color="#6366f1", slider_color_dark="#6366f1",
    button_primary_background_fill="linear-gradient(135deg,#6366f1,#7c3aed)",
    button_primary_background_fill_hover="linear-gradient(135deg,#4f46e5,#6d28d9)",
    button_primary_text_color="white", button_secondary_background_fill="#1e293b",
    button_secondary_background_fill_dark="#1e293b", button_secondary_text_color="#e5e7eb")

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

def run(mode, imodel, vmodel, prompt, style, size, count, isteps, secs, vsteps, seed, progress=gr.Progress()):
    if mode == "Image":
        imgs, hist, st = make_image(imodel, prompt, style, size, count, isteps, seed, progress)
        return gr.update(value=imgs, visible=True), gr.update(visible=False), hist, st
    path, st = make_video(vmodel, prompt, style, secs, vsteps, seed, progress)
    return gr.update(visible=False), gr.update(value=path, visible=True), history[:24], st

def switch(mode):
    img = mode == "Image"
    return (gr.update(visible=img), gr.update(visible=img), gr.update(visible=img),
            gr.update(visible=not img), gr.update(visible=not img),
            gr.update(visible=img), gr.update(visible=not img),
            gr.update(value="Generate image" if img else "Generate video"),
            gr.update(visible=img), gr.update(visible=not img))

print("2/3 Building the studio ...")
with gr.Blocks(title="Studio", theme=theme, css=CSS) as app:
    gr.HTML(f"""<div id="topbar">
      <div class="brand"><span class="mark"></span>Studio<small>Image &amp; Video Generation</small></div>
      <div class="badges"><span class="badge ok">{gpu}</span><span class="badge">SDXL · FLUX</span><span class="badge">LTX · Wan 2.1</span><span class="badge">Free session</span></div>
    </div>""")
    with gr.Row(equal_height=False):
        with gr.Column(scale=2, min_width=340, elem_id="panel"):
            mode = gr.Radio(["Image", "Video"], value="Image", label="Mode")
            imodel = gr.Dropdown(["SDXL", "FLUX.1 schnell"], value="SDXL", label="Model",
                info="SDXL: fast (~30 s). FLUX: sharper, ~1-2 min, first load ~10 min")
            vmodel = gr.Dropdown(["LTX-Video", "Wan 2.1 1.3B"], value="LTX-Video", label="Model", visible=False,
                info="LTX: 2-5 min. Wan: better motion, 10-20 min on the free GPU")
            prompt = gr.Textbox(elem_id="prompt", label="Prompt", lines=4,
                placeholder="Describe the scene, subject, lighting and mood…")
            style = gr.Dropdown(list(STYLES), value="Photorealistic", label="Style preset")
            size = gr.Dropdown(list(SIZES), value="Square 1:1", label="Aspect ratio")
            count = gr.Slider(1, 4, value=1, step=1, label="Number of images")
            secs = gr.Slider(2, 5, value=4, step=1, label="Length (seconds)", visible=False)
            with gr.Accordion("Advanced", open=False):
                isteps = gr.Slider(15, 40, value=25, step=1, label="Quality (steps)")
                vsteps = gr.Slider(20, 40, value=30, step=1, label="Quality (steps)", visible=False)
                seed = gr.Number(value=-1, label="Seed  (-1 = random)", precision=0)
            go = gr.Button("Generate image", variant="primary", elem_id="go")
            status = gr.Markdown("Ready.", elem_id="status")
        with gr.Column(scale=5, elem_id="canvas"):
            gr.HTML('<div class="section-title">Result</div>')
            igal = gr.Gallery(show_label=False, columns=2, height=560, object_fit="contain", show_download_button=True, preview=True)
            vout = gr.Video(show_label=False, height=560, visible=False)
            gr.HTML('<div class="section-title" style="margin-top:12px">Recent</div>')
            hgal = gr.Gallery(show_label=False, columns=8, height=120, object_fit="cover", elem_id="recent", allow_preview=True)
    mode.change(switch, mode, [size, count, isteps, secs, vsteps, imodel, vmodel, go, igal, vout])
    inputs = [mode, imodel, vmodel, prompt, style, size, count, isteps, secs, vsteps, seed]
    go.click(run, inputs, [igal, vout, hgal, status])
    prompt.submit(run, inputs, [igal, vout, hgal, status])

print("3/3 Ready. Use the studio below (backup link printed underneath).")
app.queue().launch(share=True, inline=True, debug=False)
